# Prédiction du churn client – VoltEdge

## Contexte

Ce cas d'étude a pour objectif de prédire le risque de départ des clients d'un fournisseur d'énergie à l'aide d'un modèle de régression logistique.

L'analyse repose sur les caractéristiques des clients telles que l'âge, l'ancienneté, la consommation annuelle et le nombre de factures impayées.

L'objectif métier est d'identifier les clients présentant un risque élevé de churn afin de pouvoir mettre en place des actions de fidélisation ciblées.

### 1. Importation des bibliothèques nécessaires

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

### 2.1 Chargement des données

In [2]:
data = pd.read_excel("../DATA/INPUT/clients_voltedge.xlsx")

In [3]:
data.head()

,Age,AncienneteMois,ConsommationAnnuelle,FacturesImpayeDernierAn,Churn
0,56,26,13485,0,1
1,69,85,14056,0,1
2,46,86,7194,0,0
3,32,7,5909,0,1
4,60,14,5715,0,0


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Age                      2000 non-null   int64
 1   AncienneteMois           2000 non-null   int64
 2   ConsommationAnnuelle     2000 non-null   int64
 3   FacturesImpayeDernierAn  2000 non-null   int64
 4   Churn                    2000 non-null   int64
dtypes: int64(5)
memory usage: 78.3 KB


In [5]:
data['Churn'] = data['Churn'].astype('category')

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   Age                      2000 non-null   int64   
 1   AncienneteMois           2000 non-null   int64   
 2   ConsommationAnnuelle     2000 non-null   int64   
 3   FacturesImpayeDernierAn  2000 non-null   int64   
 4   Churn                    2000 non-null   category
dtypes: category(1), int64(4)
memory usage: 64.7 KB


In [7]:
data.describe().round()

,Age,AncienneteMois,ConsommationAnnuelle,FacturesImpayeDernierAn
count,2000.0,2000.0,2000.0,2000.0
mean,50.0,60.0,8085.0,1.0
std,18.0,35.0,3006.0,1.0
min,18.0,1.0,1000.0,0.0
25%,34.0,29.0,6045.0,0.0
50%,50.0,60.0,8108.0,0.0
75%,66.0,90.0,10117.0,1.0
max,80.0,120.0,17781.0,6.0


La variable cible est répartie entre **1 200 clients restés chez VoltEdge (60 %)** et **800 clients ayant quitté l'entreprise (40 %)**.

In [8]:
data["Churn"].value_counts()

Churn
0    1200
1     800
Name: count, dtype: int64

### 2.2 Sélection des Features et de la Target

In [9]:
X = data[['Age', 'AncienneteMois', 'ConsommationAnnuelle', 'FacturesImpayeDernierAn']]
y = data['Churn']  # Churn : 1 si le client a quitté VoltEdge, 0 si est resté client

### 3. Prétraitement des données

In [10]:
# Division des données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

- train_test_split est une fonction qui divise les données en deux parties : une pour entraîner le modèle (80% des données dans ce cas) et l'autre pour tester le modèle (20% des données).
- test_size=0.2 signifie que 20% des données sont utilisées comme ensemble de test.
- random_state=42 assure que la division des données est reproductible; c'est-à-dire que vous obtenez la même division chaque fois que vous exécutez ce code.

### 4. Entraînement du modèle

In [11]:
model = LogisticRegression()
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


LogisticRegression() crée un modèle de régression logistique.
- model.fit(X_train, y_train) entraîne ce modèle en utilisant l'ensemble d'entraînement (X_train pour les caractéristiques et y_train pour l'étiquette cible).

### 5. Évaluation du modèle


In [12]:
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f'Précision du modèle: {accuracy * 100:.2f}%')

Précision du modèle: 82.75%


model.predict(X_test) utilise le modèle entraîné pour faire des prédictions sur l'ensemble de test (X_test).
- accuracy_score(y_test, predictions) calcule la précision du modèle, c'est-à-dire le pourcentage de prédictions correctes par rapport à l'ensemble de test.
- print(f'Précision du modèle: {accuracy * 100:.2f}%') affiche la précision du modèle en pourcentage.

In [13]:
# Création de la matrice de confusion
conf_matrix = confusion_matrix(y_test, predictions)

In [14]:
# Affichage de la matrice de confusion
print('Matrice de confusion :')
print(conf_matrix)

Matrice de confusion :
[[215  30]
 [ 39 116]]


### Lecture de la matrice de confusion

- **Vrais négatifs (TN) = 215 :** le modèle a correctement prédit 215 clients qui sont restés chez VoltEdge.

- **Faux positifs (FP) = 30 :** le modèle a prédit un départ pour 30 clients qui, en réalité, sont restés.

- **Faux négatifs (FN) = 39 :** le modèle a prédit que 39 clients resteraient alors qu'en réalité, ils ont quitté VoltEdge.

- **Vrais positifs (TP) = 116 :** le modèle a correctement identifié 116 clients qui ont quitté VoltEdge.

### Interprétation simple

Le modèle réalise **331 bonnes prédictions sur 400**, soit une précision globale de **82,75 %**.

Il identifie correctement **116 départs** et **215 non-départs**.

On remarque cependant qu'il produit davantage de **faux négatifs (39)** que de **faux positifs (30)**. Le modèle a donc plus tendance à manquer un départ réel qu'à prédire à tort qu'un client va partir.

Dans l'ensemble, les résultats sont corrects, mais le modèle pourrait encore être amélioré afin de réduire ces erreurs.

### 6. Prédictions : Création de la fonction predict_churn()

In [15]:
def predict_churn(age, anciennete_mois, consommation_annuelle, factures_impaye):

    features = pd.DataFrame([[
        age,
        anciennete_mois,
        consommation_annuelle,
        factures_impaye
    ]], columns=[
        'Age',
        'AncienneteMois',
        'ConsommationAnnuelle',
        'FacturesImpayeDernierAn'
    ])

    proba_churn = model.predict_proba(features)[0][1]

    return f"Probabilité de churn : {proba_churn * 100:.2f}%"

In [16]:
print(predict_churn(25, 6, 15000, 2))

Probabilité de churn : 99.60%


### Explication du code permettant de créer la fonction predict_churn()

L'objectif de la fonction `predict_churn` est de prédire la probabilité qu'un client quitte VoltEdge à partir de son âge, de son ancienneté, de sa consommation annuelle et du nombre de factures impayées.

- `def predict_churn(age, anciennete_mois, consommation_annuelle, factures_impaye)` définit la fonction et les quatre caractéristiques utilisées pour réaliser la prédiction.

- `features = pd.DataFrame(...)` rassemble les caractéristiques du client dans un DataFrame avec les mêmes noms de colonnes que ceux utilisés pour entraîner le modèle.

- `model.predict_proba(features)[0][1]` calcule les probabilités prédites par le modèle. `[0][1]` permet de récupérer la probabilité de la classe 1, c'est-à-dire la probabilité de churn.

- Le `return` transforme cette probabilité en pourcentage avec deux décimales.

Pour ce client, le modèle estime la probabilité de churn à **99,60 %**. Il présente donc un risque de départ très élevé et pourrait faire partie des clients à cibler en priorité par une action de fidélisation.

# Application du modèle sur les nouveaux clients

In [17]:
# Charger les données des nouveaux clients pour lesquels nous souhaitons prédire le risque de churn
new_customers = pd.read_excel("../DATA/INPUT/new_clients_voltedge.xlsx")

new_customers.head(10)

,Age,AncienneteMois,ConsommationAnnuelle,FacturesImpayeDernierAn
0,63,49,1000,1
1,67,42,10010,0
2,20,27,10956,1
3,79,39,3242,1
4,39,83,8529,1
5,36,25,9353,0
6,52,38,7780,1
7,29,85,5627,1
8,71,24,11047,1
9,22,24,6789,0


In [18]:
# Prédire la probabilité de churn des nouveaux clients
proba_churn = model.predict_proba(
    new_customers[['Age', 'AncienneteMois', 'ConsommationAnnuelle', 'FacturesImpayeDernierAn']])[:, 1]

# Ajouter les probabilités de churn au DataFrame
new_customers['ProbabiliteChurn'] = proba_churn

# Afficher les résultats
new_customers.head(20)

,Age,AncienneteMois,ConsommationAnnuelle,FacturesImpayeDernierAn,ProbabiliteChurn
0,63,49,1000,1,0.102776
1,67,42,10010,0,0.154053
2,20,27,10956,1,0.950996
3,79,39,3242,1,0.085544
4,39,83,8529,1,0.573658
5,36,25,9353,0,0.552750
6,52,38,7780,1,0.555126
7,29,85,5627,1,0.538914
8,71,24,11047,1,0.539810
9,22,24,6789,0,0.605978


Nous avons appliqué le modèle aux nouveaux clients afin d'estimer leur probabilité de churn.

Une colonne `ProbabiliteChurn` a été ajoutée au DataFrame afin d'identifier les clients présentant le risque de départ le plus élevé.

Le fichier contenant les résultats est ensuite exporté au format Excel.

In [20]:
new_customers.to_excel("../DATA/OUTPUT/pred_churn_new_clients_Benyahia_Lounis.xlsx", index=False)

## Conclusion

Le modèle de régression logistique obtient une précision de **82,75 %** sur les données de test.

La matrice de confusion montre que le modèle identifie correctement une grande partie des clients qui restent et des clients qui quittent VoltEdge. Il produit cependant davantage de faux négatifs que de faux positifs, ce qui signifie que certains clients à risque ne sont pas détectés.

Le modèle peut donc constituer une première aide pour identifier les clients susceptibles de partir et orienter des actions de fidélisation ciblées.